# GradeSense: Fine-tuning DeBERTa-v3 on SciEntsBank (Google Colab)

Fine-tunes `microsoft/deberta-v3-base` as a **cross-encoder** for 5-way short-answer grading.

**Run on Google Colab with a GPU:** Runtime → Change runtime type → T4 GPU. Training takes about 35 minutes (10 epochs).

| Setting | Value |
|---|---|
| Input | `[CLS] question + reference [SEP] student answer [SEP]`, max 256 tokens |
| Optimiser | AdamW, lr 2e-5, weight decay 0.01 (none on bias/LayerNorm) |
| Schedule | linear warmup (10% of steps), then linear decay |
| Batch size / epochs | 16 / 10 |
| Loss | class-weighted cross-entropy (weights capped at 10) |
| Model selection | best validation macro-F1 (early stopping by checkpoint) |

Splits match `src/data.py`: 10% of the official training split (stratified, seed 42) is used for validation.

**Outputs** (saved to Google Drive, then copied into the project's `models/` and `results/` folders):
`deberta_5way/` (model), `results_deberta_5way.csv`, `preds_deberta_5way_<split>.csv`, `history_deberta_5way.csv`.

## 1. Check the GPU and install packages

In [ ]:
!nvidia-smi
!pip install -q sentencepiece protobuf

## 2. Connect Google Drive (to keep the trained model)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/GradeSense', exist_ok=True)
print("Drive ready")

## 3. Settings and data (same splits as the laptop scripts)

In [ ]:
import os, time, random
import numpy as np, pandas as pd, torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

SCHEME = "5way"
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN, BATCH, EPOCHS, LR, WARMUP, SEED = 256, 16, 10, 2e-5, 0.1, 42
DRIVE_DIR = "/content/drive/MyDrive/GradeSense"
SAVE_DIR = f"{DRIVE_DIR}/deberta_{SCHEME}"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| Epochs:", EPOCHS)

LABELS_5WAY = ["correct", "contradictory", "partially_correct_incomplete", "irrelevant", "non_domain"]
LABEL_SCHEMES = {
    "5way": {n: n for n in LABELS_5WAY},
    "3way": {"correct": "correct", "contradictory": "contradictory", "partially_correct_incomplete": "incorrect",
             "irrelevant": "incorrect", "non_domain": "incorrect"},
    "2way": {"correct": "correct", "contradictory": "incorrect", "partially_correct_incomplete": "incorrect",
             "irrelevant": "incorrect", "non_domain": "incorrect"},
}
SCHEME_LABELS = {"5way": LABELS_5WAY, "3way": ["correct", "contradictory", "incorrect"], "2way": ["correct", "incorrect"]}
LABELS = SCHEME_LABELS[SCHEME]

raw = load_dataset("nkazi/SciEntsBank")
names5 = raw["train"].features["label"].names
name_to_id = {n: i for i, n in enumerate(LABELS)}

def to_df(split):
    df = raw[split].to_pandas()
    df["label_name"] = df["label"].map(lambda i: LABEL_SCHEMES[SCHEME][names5[i]])
    df["label"] = df["label_name"].map(name_to_id)
    return df

full_train = to_df("train")
tr, va = train_test_split(full_train, test_size=0.1, stratify=full_train["label"], random_state=SEED)
splits = {"train": tr.reset_index(drop=True), "val": va.reset_index(drop=True)}
for s in ["test_ua", "test_uq", "test_ud"]:
    splits[s] = to_df(s)
for k, v in splits.items():
    print(f"{k:8s}: {len(v)} rows")

## 4. Tokenizer and batching (cross-encoder input)

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_batches(df, shuffle):
    """Input format: [CLS] question + reference [SEP] student answer [SEP]"""
    order = np.random.permutation(len(df)) if shuffle else np.arange(len(df))
    for i in range(0, len(df), BATCH):
        b = df.iloc[order[i:i + BATCH]]
        enc = tokenizer(list(b["question"] + " " + b["reference_answer"]), list(b["student_answer"]),
                        truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor(b["label"].values)
        yield enc

example = next(make_batches(splits["train"].head(1), False))
print(tokenizer.decode(example["input_ids"][0]))

## 5. Fine-tune

`.float()` matters: the downloaded checkpoint is stored in half precision, and training in fp16 without a scaler
fails with *"expected scalar type Half but found Float"* in the loss.

In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS)).float().to(device)

w = compute_class_weight("balanced", classes=np.arange(len(LABELS)), y=splits["train"]["label"].values)
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(np.clip(w, None, 10.0), dtype=torch.float32, device=device))

no_decay = ["bias", "LayerNorm.weight"]
optimizer = torch.optim.AdamW([
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": 0.01},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
], lr=LR)
total_steps = EPOCHS * int(np.ceil(len(splits["train"]) / BATCH))
warm_steps = int(WARMUP * total_steps)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: (s + 1) / max(1, warm_steps) if s < warm_steps
    else max(0.0, (total_steps - s) / max(1, total_steps - warm_steps)))

@torch.no_grad()
def predict(df):
    model.eval()
    preds, probs = [], []
    for enc in make_batches(df, shuffle=False):
        enc.pop("labels")
        logits = model(**{k: v.to(device) for k, v in enc.items()}).logits
        p = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        probs.append(p); preds.append(p.argmax(1))
    return np.concatenate(preds), np.concatenate(probs)

def macro_f1(df, p):
    return f1_score(df["label"], p, labels=list(range(len(LABELS))), average="macro", zero_division=0)

print("Model dtype:", next(model.parameters()).dtype)

best_f1, best_state, history = -1, None, []
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); running = 0.0
    for step, enc in enumerate(make_batches(splits["train"], shuffle=True), 1):
        labels = enc.pop("labels").to(device)
        logits = model(**{k: v.to(device) for k, v in enc.items()}).logits
        loss = loss_fn(logits.float(), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        running += loss.item()
        if step % 50 == 0:
            print(f"  epoch {epoch} step {step:3d} | loss {running / step:.4f}")
    val_pred, _ = predict(splits["val"])
    vf1 = macro_f1(splits["val"], val_pred)
    history.append((epoch, running / step, vf1))
    tag = ""
    if vf1 > best_f1:
        best_f1, tag = vf1, "  <- best"
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"Epoch {epoch} | train loss {running / step:.4f} | val macro-F1 {vf1:.4f} | {time.time() - t0:.0f}s{tag}")

print(f"\nBest val macro-F1: {best_f1:.4f}")

## 6. Evaluate on all splits and save to Drive

In [ ]:
model.load_state_dict(best_state)
model.save_pretrained(SAVE_DIR); tokenizer.save_pretrained(SAVE_DIR)
print("Model saved to", SAVE_DIR)

rows = []
for split in ["val", "test_ua", "test_uq", "test_ud"]:
    df = splits[split]
    p, probs = predict(df)
    y = df["label"].values
    rows.append({"model": "DeBERTa-v3", "scheme": SCHEME, "split": split,
                 "accuracy": round(accuracy_score(y, p), 4),
                 "macro_f1": round(macro_f1(df, p), 4),
                 "weighted_f1": round(f1_score(y, p, labels=list(range(len(LABELS))), average="weighted", zero_division=0), 4)})
    pd.DataFrame({"id": df["id"], "label": y, "pred": p}).to_csv(f"{DRIVE_DIR}/preds_deberta_{SCHEME}_{split}.csv", index=False)
    if split != "val":
        print(f"\n--- {split} ---")
        print(classification_report(y, p, labels=list(range(len(LABELS))), target_names=LABELS, zero_division=0, digits=3))

res = pd.DataFrame(rows)
res.to_csv(f"{DRIVE_DIR}/results_deberta_{SCHEME}.csv", index=False)
pd.DataFrame(history, columns=["epoch", "train_loss", "val_macro_f1"]).to_csv(f"{DRIVE_DIR}/history_deberta_{SCHEME}.csv", index=False)
print(res.to_string(index=False))

## 7. Demo check: negation and paraphrase

In [ ]:
ref = "When the string is tighter, the pitch will be higher."
q = "What happens to the pitch when the string is tightened?"
answers = ["The pitch will be lower.",
           "A tighter string makes a higher sound.",
           "The pitch goes up when you tighten it.",
           "The string is made of nylon."]
demo = pd.DataFrame({"question": q, "reference_answer": ref, "student_answer": answers, "label": 0})
p, probs = predict(demo)
for a, pi, pr in zip(answers, p, probs):
    print(f"{a:42s} -> {LABELS[pi]:30s} ({pr[pi]:.0%})")

## 8. Publish the model to the Hugging Face Hub (for the web app)

Run once after training. The web app can download the model from this repo (`model_repo` setting).
No GPU needed for this cell.

In [ ]:
from huggingface_hub import login, HfApi
login()
from transformers import AutoTokenizer, AutoModelForSequenceClassification
repo = HfApi().whoami()["name"] + "/gradesense-deberta-5way"
AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).float().push_to_hub(repo, private=False)
AutoTokenizer.from_pretrained(SAVE_DIR).push_to_hub(repo, private=False)
print("UPLOADED:", repo)

## Results obtained (10 epochs, best epoch 7)

| Split | Accuracy | Macro-F1 | Weighted-F1 |
|---|---|---|---|
| Validation | 0.757 | 0.730 | 0.757 |
| Unseen Answers | 0.693 | 0.539 | 0.698 |
| Unseen Questions | 0.563 | 0.395 | 0.547 |
| Unseen Domains | 0.574 | 0.596 | 0.575 |